In [1]:
import pandas as pd
from pathlib import Path
import math
import sys
import os

In [12]:
sys.path.append(os.path.abspath("../src"))
from utils.split_map import create_split_map
from utils.data import compute_fingerprint_column
from utils.consts import CLEANED_DATA_LOCATION, DATA_FILE_PREFIX, BASELINE_DATA_LOCATION, SPLIT_MAP_NAME
from utils.path import ensure_dir

In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 200)

In [4]:
folder = Path(f"../{CLEANED_DATA_LOCATION}")
parquet_files = list(folder.glob("*.parquet"))

df = pd.DataFrame()

for chunk in parquet_files:
    df_chunk = pd.read_parquet(chunk)
    df = pd.concat([df, df_chunk], ignore_index=True)
    del df_chunk

In [5]:
df.head()

,activity_id,ic50,smiles,assay_type,confidence_score,target_id,chembl_id,target_type,organism,mw_freebase,alogp,hba,hbd,psa,rtb,ro3_pass,num_ro5_violations,full_mwt,aromatic_rings,heavy_atoms,qed_weighted,np_likeness_score,pic50
0,56228,690.0,N#C/C=C1\CCN(CC2CCCCN2C(=O)Cc2ccc(Cl)c(Cl)c2)C1,F,8,137,CHEMBL237,SINGLE PROTEIN,Homo sapiens,392.33,4.07,3.0,0.0,47.34,4.0,0,0.0,392.33,1.0,26.0,0.73,-0.99,6.161151
1,66179,78.0,CC(C)=CCN1CCn2c(=S)[nH]c3cccc(c32)C1C,B,9,228,CHEMBL247,SINGLE PROTEIN,Human immunodeficiency virus 1,287.43,4.04,3.0,1.0,23.96,2.0,0,0.0,287.43,2.0,20.0,0.67,-0.27,7.107905
2,68682,145.0,CC(C)=CCN1CCn2c(=O)[nH]c3ccc(Cl)c(c32)C1C,B,9,228,CHEMBL247,SINGLE PROTEIN,Human immunodeficiency virus 1,305.81,3.33,3.0,1.0,41.03,2.0,0,0.0,305.81,2.0,21.0,0.86,-0.15,6.838632
3,70002,48100.0,C=C(C)CN1Cc2cccc3[nH]c(=O)n(c23)C(CCC)C1,B,9,228,CHEMBL247,SINGLE PROTEIN,Human immunodeficiency virus 1,285.39,3.06,3.0,1.0,41.03,4.0,0,0.0,285.39,2.0,21.0,0.88,-0.57,4.317855
4,72610,10.0,CC(C)(C)OC(=O)Nc1ccc2c(c1)CN(S(=O)(=O)c1ccc(-c...,B,8,11110,CHEMBL4588,SINGLE PROTEIN,Homo sapiens,541.60,4.46,6.0,3.0,125.04,5.0,0,1.0,541.60,3.0,38.0,0.32,-1.32,8.000000


# Extracting quality rows

In [6]:
df_quality = df[
    (df['assay_type'] == 'B') &
    (df['confidence_score'] > 6) &
    (df['pic50'] > 2) &
    (df['pic50'] < 13) &
    (df['target_type'] == 'SINGLE PROTEIN') &
    (df['organism'] == 'Homo sapiens')
].copy()

# Creating baseline dataset

In [7]:
df_baseline = df_quality[['activity_id', 'smiles', 'pic50']].copy()

In [8]:
del df
del df_quality

In [8]:
df_baseline.head()

,activity_id,smiles,pic50
4,72610,CC(C)(C)OC(=O)Nc1ccc2c(c1)CN(S(=O)(=O)c1ccc(-c...,8.000000
5,73321,CC(=N)NCCOC[C@H](N)C(=O)O,5.920819
7,149048,N#Cc1ccc2c(c1)NC(=O)C2c1ncnc2cc(OCCCN3CCOCC3)c...,5.397940
12,163200,O=C(O)CN1C(=O)[C@@H](NC(=O)[C@@H](S)Cc2ccccc2)...,7.886057
19,358498,CC[C@@H]1/C=C(\C)C[C@H](C)C[C@H](OC)[C@H]2O[C@...,8.552842


In [10]:
CHUNK_SIZE = 50_000
ensure_dir(f"../{BASELINE_DATA_LOCATION}")

for i in range(math.ceil(len(df_baseline) / CHUNK_SIZE)):
    chunk = df_baseline.iloc[i*CHUNK_SIZE:(i+1)*CHUNK_SIZE]
    compute_fingerprint_column(chunk)
    chunk.to_parquet(f"../{BASELINE_DATA_LOCATION}/{DATA_FILE_PREFIX}{i}.parquet", index=False)

In [13]:
create_split_map(f"../{BASELINE_DATA_LOCATION}", DATA_FILE_PREFIX, SPLIT_MAP_NAME)

In [43]:
df_split = pd.read_parquet(f"{LOCATION}/split_map.parquet")
df_split.head()

,activity_id,split
0,24456824,train
1,24456826,train
2,24670263,val
3,24754077,train
4,24817100,train
